In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# This script provides the masked data for infilling based downscalling.
# It needs file with original grid (e.g. 0.1 degree) and file with upscaled data (e.g., 1 degree)
# Mask: 0 for points to be infilled, 1 for values to use for infilling.
# Masked_data: HR grid with HR values, except mask==1 where LR data is inserted 

In [22]:
year_min = 1950
year_max = 2024
year_ref_min = 1985
year_ref_max = 2014
var = 't2max'

In [4]:
# Only keep years with 365 days, remove leap years
def remove_leap_days(ds):
    # Get day-of-year for each time step
    day_of_year = ds['time'].dt.dayofyear
    # Keep only days <= 365 (i.e., remove Feb 29)
    ds_corrected = ds.sel(time=day_of_year <= 365)
    return ds_corrected

# Compute anomalies

In [13]:
# Compute climatology
#load data
path_in = '/work/uo1075/u241308/data_python_PostDoc/BB_challenge/data/era5_land/'+var+'/highres/'
files_in = [path_in + f"era5_land_{var}_daily_{y}_europe_original.nc" for y in range(year_ref_min, year_ref_max + 1)]

data_clim = xr.open_mfdataset(files_in,combine="by_coords",chunks={"valid_time": 365, "lat": 200, "lon": 200})

#change time dimension name
data_clim = data_clim.rename({'valid_time': 'time'})

# remove Feb 29
data_clim = remove_leap_days(data_clim) #leap years to 365

# compute climatology
climatology = data_clim.groupby("time.dayofyear").mean("time")

# load into memory (important!)
climatology = climatology.compute()

# optionally save it
path_clim = '/work/uo1075/u241308/data_python_PostDoc/BB_challenge/data/era5_land/'+var+'/highres/'
file_clim = [f'era5_land_{var}_daily_europe_original_climatology_ref_{year_ref_min}_{year_ref_max}.nc']
climatology.to_netcdf("path_clim+file_clim")

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/xarray/core/concat.py:546: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/xarray/core/concat.py:546: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


In [23]:
climatology = xr.open_dataset("path_clim+file_clim")

path_in = '/work/uo1075/u241308/data_python_PostDoc/BB_challenge/data/era5_land/'+var+'/highres/'
for year in range(year_min, year_max+1):
    print([f'Starting year {year}'])
    file_in = f"{path_in}era5_land_{var}_daily_{year}_europe_original.nc"
    data_original = xr.open_dataset(file_in, chunks={"time": 365, "lat": 200, "lon": 200})

    #change time dimension name
    data_original = data_original.rename({'valid_time': 'time'})
    
    # remove Feb 29
    data_original = remove_leap_days(data_original) #leap years to 365

    #compute anomalies
    anomalies = data_original.groupby("time.dayofyear") - climatology
    anomalies = anomalies.drop_vars("dayofyear")

    #save data
    path_out = '/work/uo1075/u241308/data_python_PostDoc/BB_challenge/data/era5_land/'+var+'/highres/anomalies/'
    file_out = f"{path_out}era5_land_{var}_daily_{year}_europe_anomalies_ref_{year_ref_min}_{year_ref_max}.nc"
    anomalies.to_netcdf(file_out)

['Starting year 1950']
['Starting year 1951']
['Starting year 1952']
['Starting year 1953']
['Starting year 1954']
['Starting year 1955']
['Starting year 1956']
['Starting year 1957']
['Starting year 1958']
['Starting year 1959']
['Starting year 1960']
['Starting year 1961']
['Starting year 1962']
['Starting year 1963']
['Starting year 1964']
['Starting year 1965']
['Starting year 1966']
['Starting year 1967']
['Starting year 1968']
['Starting year 1969']
['Starting year 1970']
['Starting year 1971']
['Starting year 1972']
['Starting year 1973']
['Starting year 1974']
['Starting year 1975']
['Starting year 1976']
['Starting year 1977']
['Starting year 1978']
['Starting year 1979']
['Starting year 1980']
['Starting year 1981']
['Starting year 1982']
['Starting year 1983']
['Starting year 1984']
['Starting year 1985']
['Starting year 1986']
['Starting year 1987']
['Starting year 1988']
['Starting year 1989']
['Starting year 1990']
['Starting year 1991']
['Starting year 1992']
['Starting 